# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import os

data_path = './data'


In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

print("Training data sample:")
print(df_train[['src', 'tgt']].head())
print(f"\nDataset sizes: Train={len(df_train)}, Eval={len(df_eval)}")


Training data sample:
           src   tgt
0  14*(43+20)=   882
1     (6+1)*5=    35
2    13+32+29=    74
3   31*(3-11)=  -248
4     24*49+1=  1177

Dataset sizes: Train=2369250, Eval=263250


# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [4]:
char_to_id = {}
id_to_char = {}

# Add special tokens
char_to_id['<pad>'] = 0
char_to_id['<sos>'] = 1  # Start of sequence
char_to_id['<eos>'] = 2  # End of sequence
char_to_id['<unk>'] = 3  # Unknown character

id_to_char[0] = '<pad>'
id_to_char[1] = '<sos>'
id_to_char[2] = '<eos>'
id_to_char[3] = '<unk>'

# Build vocabulary from all characters in training data
idx = 4  # Start from 4 now
all_chars = ''.join(df_train['src'])
for char in sorted(set(all_chars)):
    if char not in char_to_id:
        char_to_id[char] = idx
        id_to_char[idx] = char
        idx += 1

vocab_size = len(char_to_id)
print(f'Vocab size: {vocab_size}')
print('Character mappings (first 20):')
for i in range(min(20, vocab_size)):
    print(f'  {id_to_char[i]}: {i}')


Vocab size: 20
Character mappings (first 20):
  <pad>: 0
  <sos>: 1
  <eos>: 2
  <unk>: 3
  (: 4
  ): 5
  *: 6
  +: 7
  -: 8
  0: 9
  1: 10
  2: 11
  3: 12
  4: 13
  5: 14
  6: 15
  7: 16
  8: 17
  9: 18
  =: 19


In [5]:
# ============================================================
# Optimized Hyperparameters for Full Dataset Training
# ============================================================

# Training Configuration
batch_size = 1024       # 16x larger! (64 → 1024) - utilize GPU memory
epochs = 10             # Reduced → faster convergence with large batches
lr = 0.003              # Increased to match larger batch size

# Model Architecture  
embed_dim = 512         
hidden_dim = 512        
num_layers = 3          
dropout_rate = 0.3      

# Regularization
grad_clip = 5.0         # Less aggressive clipping
label_smoothing = 0.1   # prevent overconfidence

# Learning Rate Scheduler
use_lr_scheduler = True
lr_scheduler_patience = 2
lr_scheduler_factor = 0.5

# Data Loading
num_workers = 4         # Parallel data loading

print("=" * 70)
print(f"  Batch Size:      {batch_size:>6}")
print(f"  Epochs:          {epochs:>6}")
print(f"  Learning Rate:   {lr:>6}")
print(f"  Embedding Dim:   {embed_dim:>6}")
print(f"  Hidden Dim:      {hidden_dim:>6}")
print(f"  LSTM Layers:     {num_layers:>6}")
print(f"  Dropout:         {dropout_rate:>6}")
print(f"  Label Smoothing: {label_smoothing:>6}")
print(f"  LR Scheduler:    {'Yes':>6}")
print(f"  Grad Clip:       {grad_clip:>6}")
print(f"  Data Workers:    {num_workers:>6}")
print("=" * 70)


  Batch Size:        1024
  Epochs:              10
  Learning Rate:    0.003
  Embedding Dim:      512
  Hidden Dim:         512
  LSTM Layers:          3
  Dropout:            0.3
  Label Smoothing:    0.1
  LR Scheduler:       Yes
  Grad Clip:          5.0
  Data Workers:         4


In [6]:
def char_to_id_with_unk(char):
    """Convert character to ID, use <unk> for unknown characters"""
    return char_to_id.get(char, char_to_id['<unk>'])

def create_training_sequences(row):
    """
    Create input/output sequences for seq2seq arithmetic learning.
    Input: equation with '=' (e.g., "1+2=")
    Output: <sos> + answer + <eos> (e.g., "<sos>3<eos>")
    """
    src = row['src']
    tgt = row['tgt']
    
    if not src.endswith('='):
        src = src + '='
    
    input_ids = [char_to_id_with_unk(c) for c in src]
    label_ids = [char_to_id['<sos>']] + [char_to_id_with_unk(c) for c in tgt] + [char_to_id['<eos>']]
    equals_pos = src.index('=')
    
    return input_ids, label_ids, equals_pos

# Create training sequences
df_train['char_id_list'], df_train['label_id_list'], df_train['equals_pos'] = \
    zip(*df_train.apply(create_training_sequences, axis=1))

# Show examples
print("=== Data Preprocessing Examples ===\n")
for idx in range(min(5, len(df_train))):
    row = df_train.iloc[idx]
    input_chars = [id_to_char[id] for id in row['char_id_list']]
    label_chars = [id_to_char[id] for id in row['label_id_list']]
    
    print(f"Example {idx + 1}:")
    print(f"  Equation: {row['src']}")
    print(f"  Answer: {row['tgt']}")
    print(f"  Input seq: {''.join(input_chars)}")
    print(f"  Label seq: {''.join(label_chars)}")
    print()

print("=== Dataset Statistics ===")
print(f"Total samples: {len(df_train)}")
print(f"Avg equation length: {df_train['char_id_list'].apply(len).mean():.2f}")
print(f"Avg answer length: {df_train['label_id_list'].apply(len).mean():.2f}")


=== Data Preprocessing Examples ===

Example 1:
  Equation: 14*(43+20)=
  Answer: 882
  Input seq: 14*(43+20)=
  Label seq: <sos>882<eos>

Example 2:
  Equation: (6+1)*5=
  Answer: 35
  Input seq: (6+1)*5=
  Label seq: <sos>35<eos>

Example 3:
  Equation: 13+32+29=
  Answer: 74
  Input seq: 13+32+29=
  Label seq: <sos>74<eos>

Example 4:
  Equation: 31*(3-11)=
  Answer: -248
  Input seq: 31*(3-11)=
  Label seq: <sos>-248<eos>

Example 5:
  Equation: 24*49+1=
  Answer: 1177
  Input seq: 24*49+1=
  Label seq: <sos>1177<eos>

=== Dataset Statistics ===
Total samples: 2369250
Avg equation length: 9.53
Avg answer length: 4.90
Avg equation length: 9.53
Avg answer length: 4.90


# Data Batching

- Use `torch.utils.data.Dataset` to create a data generation tool called `dataset`.
- Then, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.


In [7]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, index):
        row = self.sequences.iloc[index]
        x = row['char_id_list']
        y = row['label_id_list']
        equals_pos = row['equals_pos'] if 'equals_pos' in row else -1
        return x, y, equals_pos

def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_equals_pos = [data[2] for data in batch]
    
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    encoder_mask = torch.arange(pad_batch_x.size(1)).unsqueeze(0) < batch_x_lens.unsqueeze(1)
    encoder_mask = encoder_mask.float()
    
    equals_mask = torch.zeros_like(pad_batch_x, dtype=torch.float32)
    for i, pos in enumerate(batch_equals_pos):
        if pos >= 0 and pos < pad_batch_x.size(1):
            equals_mask[i, pos] = 1.0
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask


In [8]:
# Quick Testing Mode - Set to None to use full dataset
# TRAINING_SAMPLE_SIZE = 10000  # Use 10k samples for fast testing
TRAINING_SAMPLE_SIZE = None     # Use full dataset (2.37M samples)

# Sample training data if needed
if TRAINING_SAMPLE_SIZE is not None and TRAINING_SAMPLE_SIZE < len(df_train):
    df_train_sample = df_train.sample(n=TRAINING_SAMPLE_SIZE, random_state=42)
    print(f"Using SAMPLED training data: {TRAINING_SAMPLE_SIZE:,} samples (of {len(df_train):,} total)")
else:
    df_train_sample = df_train
    print(f"Using FULL training dataset: {len(df_train):,} samples")

ds_train = Dataset(df_train_sample[['char_id_list', 'label_id_list', 'equals_pos']])

dl_train = torch.utils.data.DataLoader(
    ds_train, 
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=num_workers,  # Parallel data loading
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if num_workers > 0 else False
)

batches_per_epoch = len(dl_train)
samples_per_second = len(ds_train) / (batches_per_epoch * 0.05)  # Rough estimate

print(f"DataLoader created:")
print(f"  Batches per epoch:  {batches_per_epoch:,}")
print(f"  Batch size:         {batch_size}")
print(f"  Num workers:        {num_workers}")
print(f"  Pin memory:         {dl_train.pin_memory}")
print(f"  Est. time/epoch:    ~{batches_per_epoch * 0.15 / 60:.1f} min (at batch_size={batch_size})")

# Verify with one batch
batch_x, batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask = next(iter(dl_train))
print(f"\nBatch shapes: x={batch_x.shape}, y={batch_y.shape}")

sample_x = batch_x[0][:batch_x_lens[0]]
sample_y = batch_y[0][:batch_y_lens[0]]
input_str = ''.join([id_to_char[int(id)] for id in sample_x])
label_str = ''.join([id_to_char[int(id)] for id in sample_y])

print(f"Sample: Input='{input_str}', Label='{label_str}'")


Using FULL training dataset: 2,369,250 samples
DataLoader created:
  Batches per epoch:  2,314
  Batch size:         1024
  Num workers:        4
  Pin memory:         True
  Est. time/epoch:    ~5.8 min (at batch_size=1024)
DataLoader created:
  Batches per epoch:  2,314
  Batch size:         1024
  Num workers:        4
  Pin memory:         True
  Est. time/epoch:    ~5.8 min (at batch_size=1024)

Batch shapes: x=torch.Size([1024, 11]), y=torch.Size([1024, 7])
Sample: Input='13-30-28=', Label='<sos>-45<eos>'

Batch shapes: x=torch.Size([1024, 11]), y=torch.Size([1024, 7])
Sample: Input='13-30-28=', Label='<sos>-45<eos>'


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [9]:
torch.manual_seed(2)

device = torch.device('cpu')

if torch.cuda.is_available():
    try:
        device = torch.device('cuda')
        test_tensor = torch.randn(2, 3).to(device)
        print(f"Using CUDA GPU (device: {device})")
    except Exception as e:
        print(f"CUDA initialization failed: {e}")
        device = torch.device('cpu')
        print(f"Falling back to CPU")
else:
    print(f"Using CPU (device: {device})")


Using CUDA GPU (device: cuda)


In [10]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.0):
        super(CharRNN, self).__init__()
        
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.encoder_embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=char_to_id['<pad>']
        )
        
        self.encoder_rnn = torch.nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.decoder_embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=char_to_id['<pad>']
        )
        
        self.decoder_rnn = torch.nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Add dropout before output layer
        self.output_layer = torch.nn.Sequential(
            torch.nn.Dropout(dropout),
            torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size)
        )
    
    def forward(self, encoder_input, encoder_lens, decoder_input, decoder_lens):
        """Seq2Seq forward pass with teacher forcing."""
        encoder_output, (hidden, cell) = self.encode(encoder_input, encoder_lens)
        decoder_output = self.decode(decoder_input, decoder_lens, hidden, cell)
        return decoder_output
    
    def encode(self, batch_x, batch_x_lens):
        """Encode the equation to get context."""
        batch_x = self.encoder_embedding(batch_x)
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(
            batch_x,
            batch_x_lens.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        output, (hidden, cell) = self.encoder_rnn(batch_x)
        return output, (hidden, cell)
    
    def decode(self, batch_y, batch_y_lens, hidden, cell):
        """Decode to generate answer using teacher forcing."""
        batch_y = self.decoder_embedding(batch_y)
        batch_y = torch.nn.utils.rnn.pack_padded_sequence(
            batch_y,
            batch_y_lens.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        output, _ = self.decoder_rnn(batch_y, (hidden, cell))
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(output, batch_first=True)
        output = self.output_layer(output)
        return output
    
    def generator(self, equation, max_len=20):
        """Generate answer for a given equation."""
        self.eval()
        
        if not equation:
            return ['<eos>']
        
        encoder_input = [char_to_id.get(c, char_to_id['<unk>']) for c in equation]
        encoder_tensor = torch.LongTensor([encoder_input]).to(device)
        encoder_lens = torch.LongTensor([len(encoder_input)])
        
        with torch.no_grad():
            _, (hidden, cell) = self.encode(encoder_tensor, encoder_lens)
            generated_ids = []
            decoder_input = torch.LongTensor([[char_to_id['<sos>']]]).to(device)
            
            for _ in range(max_len):
                embedded = self.decoder_embedding(decoder_input)
                output, (hidden, cell) = self.decoder_rnn(embedded, (hidden, cell))
                logits = self.output_layer(output[:, -1, :])
                next_token = torch.argmax(logits, dim=-1).item()
                
                if next_token == char_to_id['<eos>']:
                    generated_ids.append(next_token)
                    break
                
                generated_ids.append(next_token)
                decoder_input = torch.LongTensor([[next_token]]).to(device)
            
            return [id_to_char.get(ch_id, '<unk>') for ch_id in generated_ids]


In [11]:
model = CharRNN(
    vocab_size=vocab_size, 
    embed_dim=embed_dim, 
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout_rate
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created:")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Embedding dim:   {embed_dim}")
print(f"  Hidden dim:      {hidden_dim}")
print(f"  LSTM layers:     {num_layers}")
print(f"  Dropout:         {dropout_rate}")
print(f"  Total params:    {total_params:,}")
print(f"  Trainable:       {trainable_params:,}")
print(f"  Model size:      ~{total_params * 4 / 1024 / 1024:.1f} MB")


Model created:
  Vocabulary size: 20
  Embedding dim:   512
  Hidden dim:      512
  LSTM layers:     3
  Dropout:         0.3
  Total params:    12,900,884
  Trainable:       12,900,884
  Model size:      ~49.2 MB


# Training

The training loop:
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss.
        3. Use `loss.backward()` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_()` to limit the gradient values.
        5. Use `optimizer.step()` to update the model (backpropagation).
    2. After each epoch, perform validation on a sample of the eval dataset.
2. Save the best model based on validation accuracy.


In [12]:
# Loss with label smoothing (prevents overconfidence)
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=char_to_id['<pad>'],
    reduction='mean',
    label_smoothing=label_smoothing
)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Learning rate scheduler
if use_lr_scheduler:
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',              # Maximize validation EM
        factor=lr_scheduler_factor,
        patience=lr_scheduler_patience,
        min_lr=1e-6
    )
    print(f"LR Scheduler: ReduceLROnPlateau (factor={lr_scheduler_factor}, patience={lr_scheduler_patience})")
else:
    scheduler = None

print(f"Optimizer: Adam (lr={lr})")
print(f"Loss: CrossEntropyLoss (label_smoothing={label_smoothing})")
print(f"Gradient clipping: {grad_clip}")


LR Scheduler: ReduceLROnPlateau (factor=0.5, patience=2)
Optimizer: Adam (lr=0.003)
Loss: CrossEntropyLoss (label_smoothing=0.1)
Gradient clipping: 5.0


In [13]:
from tqdm import tqdm

VALIDATION_SAMPLE_SIZE = 50000  # Set to None to use full eval set

model = model.to(device)
model.train()

print("=" * 70)
print("Starting Optimized Seq2Seq Training")
print("=" * 70)
print(f"Training samples:   {len(ds_train):,}")
print(f"Validation samples: {VALIDATION_SAMPLE_SIZE if VALIDATION_SAMPLE_SIZE else len(df_eval):,}")
print(f"Batch size:         {batch_size}")
print(f"Batches/epoch:      {len(dl_train):,}")
print(f"Est. time/epoch:    ~{len(dl_train) * 0.15 / 60:.1f} min")
print("=" * 70)

i = 0
train_losses = []
best_em = 0.0
best_model_state = None

for epoch in range(1, epochs+1):
    epoch_losses = []
    
    bar = tqdm(dl_train, desc=f"Epoch {epoch}/{epochs}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask in bar:
        optimizer.zero_grad()
        
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        encoder_mask = encoder_mask.to(device)
        equals_mask = equals_mask.to(device)
        
        decoder_input = batch_y[:, :-1]
        decoder_lens = torch.clamp(batch_y_lens - 1, min=1)
        decoder_target = batch_y[:, 1:]
        
        batch_pred_y = model(batch_x, batch_x_lens, decoder_input, decoder_lens)
        
        pred_len = batch_pred_y.size(1)
        target_len = decoder_target.size(1)
        
        if pred_len != target_len:
            if pred_len > target_len:
                batch_pred_y = batch_pred_y[:, :target_len, :]
            else:
                decoder_target = decoder_target[:, :pred_len]
        
        batch_pred_y_reshaped = batch_pred_y.reshape(-1, vocab_size)
        batch_y_reshaped = decoder_target.reshape(-1)
        
        loss = criterion(batch_pred_y_reshaped, batch_y_reshaped)
        
        loss.backward()
        # Use gradient norm clipping (better than value clipping)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        
        epoch_losses.append(loss.item())
        train_losses.append(loss.item())
        
        i += 1
        if i % 50 == 0:
            avg_loss = np.mean(epoch_losses[-50:])
            bar.set_postfix(loss=f"{avg_loss:.6f}")
    
    epoch_loss = np.mean(epoch_losses)
    print(f"\n{'='*70}")
    print(f"Epoch {epoch}/{epochs} Complete")
    print(f"  Loss: {epoch_loss:.4f} (min={min(epoch_losses):.4f}, max={max(epoch_losses):.6f})")
    
    # Validation
    print(f"  Running validation...")
    model.eval()
    matched = 0
    total = 0
    error_samples = []
    
    eval_sample_size = min(VALIDATION_SAMPLE_SIZE, len(df_eval)) if VALIDATION_SAMPLE_SIZE else len(df_eval)
    df_eval_sample = df_eval.sample(n=eval_sample_size, random_state=42) if VALIDATION_SAMPLE_SIZE else df_eval
    
    with torch.no_grad():
        for _, row in tqdm(df_eval_sample.iterrows(), desc="  Validation", total=len(df_eval_sample), leave=False):
            equation = row['src']
            expected_answer = str(row['tgt'])
            
            try:
                prediction_chars = model.generator(equation)
                predicted_answer = ''.join(prediction_chars).replace('<eos>', '').strip()
                
                if predicted_answer == expected_answer:
                    matched += 1
                else:
                    if len(error_samples) < 5:
                        error_samples.append((equation, expected_answer, predicted_answer))
                        
            except Exception as e:
                if len(error_samples) < 5:
                    error_samples.append((equation, expected_answer, f"Error: {str(e)}"))
            
            total += 1
    
    em = matched / total if total > 0 else 0
    print(f"  Validation EM: {em:.4f} ({matched:,}/{total:,})")
    
    if error_samples:
        print(f"  Sample errors:")
        for err_idx, (eq, expected, predicted) in enumerate(error_samples[:5], 1):
            print(f"    {err_idx}. {eq:15s} Expected: {expected:8s} Got: {predicted}")
    
    # Save best model
    if em > best_em:
        best_em = em
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ✓ New best model! EM: {best_em:.4f}")
    
    # Update learning rate
    if use_lr_scheduler and scheduler is not None:
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(em)
        new_lr = optimizer.param_groups[0]['lr']
        if old_lr != new_lr:
            print(f"  Learning rate: {old_lr:.6f} → {new_lr:.6f}")
    
    print("=" * 70)
    model.train()

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)
    print(f"\n✓ Loaded best model with EM: {best_em:.4f}")

print("\n" + "=" * 70)
print("Training Complete!")
print("=" * 70)
print(f"Final loss:     {train_losses[-1]:.4f}")
print(f"Average loss:   {np.mean(train_losses):.4f}")
print(f"Best valid EM:  {best_em:.4f} ({best_em*100:.2f}%)")
print("=" * 70)


Starting Optimized Seq2Seq Training
Training samples:   2,369,250
Validation samples: 50,000
Batch size:         1024
Batches/epoch:      2,314
Est. time/epoch:    ~5.8 min


Epoch 1/10: 100%|██████████| 2314/2314 [00:55<00:00, 41.67it/s, loss=1.020590]




Epoch 1/10 Complete
  Loss: 1.2474 (min=0.9868, max=3.626907)
  Running validation...


  Validation EM: 0.5117 (25,586/50,000)
  Sample errors:
    1. 35*(26-43)=     Expected: -595     Got: -565
    2. (46*16)-7=      Expected: 729      Got: 725
    3. 18+27*29=       Expected: 801      Got: 805
    4. (35*19)+5=      Expected: 670      Got: 680
    5. (11+42)*38=     Expected: 2014     Got: 1986
  ✓ New best model! EM: 0.5117


Epoch 2/10: 100%|██████████| 2314/2314 [00:55<00:00, 41.40it/s, loss=0.909509]



...


Epoch 9/10 Complete
  Loss: 0.7004 (min=0.6796, max=0.731588)
  Running validation...


  Validation EM: 0.9271 (46,357/50,000)
  Sample errors:
    1. (3*34)+31=      Expected: 133      Got: 131
    2. (45*43)-14=     Expected: 1921     Got: 1911
    3. 47-(33*44)=     Expected: -1405    Got: -1415
    4. 45-6*4=         Expected: 21       Got: 23
    5. 31+(2*4)=       Expected: 39       Got: 41
  ✓ New best model! EM: 0.9271


Epoch 10/10: 100%|██████████| 2314/2314 [00:55<00:00, 41.45it/s, loss=0.691315]




Epoch 10/10 Complete
  Loss: 0.6922 (min=0.6724, max=0.724310)
  Running validation...


  Validation EM: 0.9376 (46,880/50,000)
  Sample errors:
    1. 18+27*29=       Expected: 801      Got: 791
    2. (11+42)*38=     Expected: 2014     Got: 1938
    3. 2*7-7=          Expected: 7        Got: 9
    4. 47-(33*44)=     Expected: -1405    Got: -1395
    5. 48*19*1=        Expected: 912      Got: 928
  ✓ New best model! EM: 0.9376

✓ Loaded best model with EM: 0.9376

Training Complete!
Final loss:     0.6872
Average loss:   0.8299
Best valid EM:  0.9376 (93.76%)


In [14]:
print("Running full evaluation on entire eval set...")
print(f"Total samples: {len(df_eval)}")

model.eval()
matched = 0
total = 0
error_samples = []

with torch.no_grad():
    for _, row in tqdm(df_eval.iterrows(), desc="Full Evaluation", total=len(df_eval)):
        equation = row['src']
        expected_answer = str(row['tgt'])
        
        try:
            prediction_chars = model.generator(equation)
            predicted_answer = ''.join(prediction_chars).replace('<eos>', '').strip()
            
            if predicted_answer == expected_answer:
                matched += 1
            else:
                if len(error_samples) < 20:
                    error_samples.append((equation, expected_answer, predicted_answer))
        except Exception as e:
            if len(error_samples) < 20:
                error_samples.append((equation, expected_answer, f"Error: {str(e)}"))
        
        total += 1

full_em = matched / total if total > 0 else 0
print(f"\n{'='*60}")
print(f"Full Evaluation Results")
print(f"{'='*60}")
print(f"Exact Match: {full_em:.4f} ({matched}/{total})")
print(f"Accuracy: {full_em*100:.2f}%")

if error_samples:
    print(f"\n{'='*60}")
    print(f"Sample Errors (showing first 10):")
    print(f"{'='*60}")
    for idx, (eq, expected, predicted) in enumerate(error_samples[:10], 1):
        print(f"{idx:2d}. {eq:20s} | Expected: {expected:10s} | Got: {predicted}")


Running full evaluation on entire eval set...
Total samples: 263250


Full Evaluation: 100%|██████████| 263250/263250 [06:56<00:00, 632.35it/s]


Full Evaluation Results
Exact Match: 0.9382 (246989/263250)
Accuracy: 93.82%

Sample Errors (showing first 10):
 1. 43*41*3=             | Expected: 5289       | Got: 5167
 2. 11-36*29=            | Expected: -1033      | Got: -1043
 3. 49*23*34=            | Expected: 38318      | Got: 38278
 4. (33*49)-33=          | Expected: 1584       | Got: 1574
 5. (6*7)+26=            | Expected: 68         | Got: 76
 6. 9+44*3=              | Expected: 141        | Got: 143
 7. 30-13+4=             | Expected: 21         | Got: 23
 8. 3*43*42=             | Expected: 5418       | Got: 5286
 9. 11*13*8=             | Expected: 1144       | Got: 1176
10. 10*5-41=             | Expected: 9          | Got: 1
